In [1]:
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── 路径配置：确保能导入 timing_framework ──────────────────────────────────
# sys.path.insert(0, str(Path(__file__).parent))
warnings.filterwarnings("ignore")

# ── Matplotlib 后端与字体（必须在其他 import 之前设置）──────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt



plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "WenQuanYi Micro Hei",
                                    "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import pandas as pd

# ── 第三方库（带友好报错）──────────────────────────────────────────────────
try:
    import yaml
except ImportError:
    raise ImportError("请安装 PyYAML：pip install pyyaml")

try:
    import openpyxl
    from openpyxl import Workbook
    from openpyxl.styles import (
        Alignment, Border, Font, PatternFill, Side,
        numbers as xl_numbers,
    )
    from openpyxl.utils import get_column_letter
    from openpyxl.formatting.rule import ColorScaleRule
except ImportError:
    raise ImportError("请安装 openpyxl：pip install openpyxl")

In [43]:
def load_config(config_path: str | Path = "config.yaml") -> dict:
    """加载 YAML 配置文件，返回配置字典。"""
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return cfg

In [44]:
np.clip(np.array([3,2,9,4,-1,9,8,2,5,6]),0,5)

array([3, 2, 5, 4, 0, 5, 5, 2, 5, 5])

In [10]:
cfg= load_config()

In [13]:
asset_cfg  = cfg.get("asset",      {})
sig_cfg    = cfg.get("signals",    {})
eval_cfg   = cfg.get("evaluation", {})
bt_cfg     = cfg.get("backtest",   {})
out_cfg    = cfg.get("output",     {})

In [21]:
asset_name  = asset_cfg.get("name",        "未知资产")
asset_code  = asset_cfg.get("code",        "")
price_field = asset_cfg.get("price_field", "CLOSE")

workspace        = out_cfg.get("workspace_dir", "workspace")
eval_subdir      = out_cfg.get("eval_plots_subdir",    "plots/eval")
bt_subdir        = out_cfg.get("backtest_plots_subdir", "plots/backtest")
excel_filename   = out_cfg.get("excel_filename",       "timing_report.xlsx")
dpi              = int(out_cfg.get("dpi", 100))

data_cfg    = cfg.get("data", {}) # -> data_cfg: dict
price_file  = data_cfg.get("price_file",  "price_df.csv") # -> price_file: Path_str
signal_file = data_cfg.get("signal_file", "signal.csv") # -> signal_file: Path_str
# price_df, signal_df = load_data(ROOT / "data_ini", price_file, signal_file)

In [25]:
def load_data(data_dir: str | Path = "data_ini",
             price_file: str = "price_df.csv",
             signal_file: str = "signal.csv") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    从 data_ini/ 目录加载价格和信号 CSV 文件。

    文件名通过 config.yaml 的 data.price_file / data.signal_file 指定。

    返回
    ----
    (price_df, signal_df)，索引均为 DatetimeIndex。
    """
    data_dir = Path(data_dir)
    price_path  = data_dir / price_file
    signal_path = data_dir / signal_file

    if not price_path.exists():
        raise FileNotFoundError(f"价格文件不存在：{price_path.resolve()}")
    if not signal_path.exists():
        raise FileNotFoundError(f"信号文件不存在：{signal_path.resolve()}")

    price_df  = pd.read_csv(price_path,  index_col=0, parse_dates=True)
    signal_df = pd.read_csv(signal_path, index_col=0, parse_dates=True)

    price_df.index  = pd.to_datetime(price_df.index)
    signal_df.index = pd.to_datetime(signal_df.index)

    price_df  = price_df.sort_index()
    signal_df = signal_df.sort_index()

    return price_df, signal_df

In [34]:
price_df, signal_df = load_data("data_ini", price_file, signal_file)

In [28]:
asset_code

'000300.SH'

In [39]:
def extract_price_series(price_df: pd.DataFrame,
                          asset_code: str,
                          price_field: str = "CLOSE") -> pd.Series:
    """
    从多资产价格 DataFrame 中智能提取指定资产的价格序列。
    虽然建议在数据表字段按照‘datetime,asset_code’名称严格设置，但这里依旧给予了几项可能的匹配策略。

    """
    #策略main: 如果结构式 datetime-code-CLOSE，直接提取
    code_columns_name = ['code','asset_code','wind_code','sec_code']
    for code_col in code_columns_name:
        if code_col in price_df.columns:
            print(f"匹配资产代码列名：{code_col}")
            price_df = price_df[price_df[code_col] == asset_code]
            return price_df[price_field].dropna().rename("price")


    # 策略1：如果结构是 datetime-code1-code2-code3
    if asset_code in price_df.columns:
        print(f"直接匹配资产代码列：{asset_code}")
        return price_df[asset_code].dropna().rename("price")

    # 策略2：如果结构是 datetime-code1_CLOSE-code2-code3
    col2 = f"{asset_code}_{price_field}"
    if col2 in price_df.columns:
        print(f"匹配资产代码和字段名：{col2}")
        return price_df[col2].dropna().rename("price")

    # 策略3：单资产，结构为 datetime-CLOSE
    pf_lower = price_field.lower()
    exact_matches = [c for c in price_df.columns
                     if isinstance(c, str) and c.lower() == pf_lower]
    if len(exact_matches) == 1:
        print(f"直接匹配字段名（大小写敏感）：{exact_matches[0]}")
        return price_df[exact_matches[0]].dropna().rename("price")

    # 策略4：MultiIndex 列
    if isinstance(price_df.columns, pd.MultiIndex):
        if (asset_code, price_field) in price_df.columns:
            print(f"匹配 MultiIndex 资产代码和字段名：{asset_code, price_field}")
            return price_df[(asset_code, price_field)].dropna().rename("price")
        if (asset_code, price_field.lower()) in price_df.columns:
            print(f"匹配 MultiIndex 资产代码和字段名（大小写敏感）：{asset_code, price_field.lower()}")
            return price_df[(asset_code, price_field.lower())].dropna().rename("price")

    # 策略5：大小写不敏感模糊匹配（含资产代码和字段名）
    for col in price_df.columns:
        if isinstance(col, str):
            cl = col.lower()
            if asset_code.lower() in cl and pf_lower in cl:
                print(f"大小写不敏感模糊匹配：{col}")
                return price_df[col].dropna().rename("price")

    # 均未匹配，给出明确的错误信息
    raise ValueError(
        f"无法在 price_df 中找到资产 '{asset_code}' 的 '{price_field}' 价格列。\n"
        f"请检查 config.yaml 的 asset.code 和 asset.price_field 设置。\n"
        f"price_df 现有列（前20个）：{list(price_df.columns[:20])}"
    )

In [40]:
prices  = extract_price_series(price_df, asset_code, price_field) # -> prices: Series

匹配资产代码列名：code


In [41]:
prices

2015-01-01     3000.000000
2015-01-02     2998.117711
2015-01-05     3023.454835
2015-01-06     3081.821852
2015-01-07     3081.005308
                  ...     
2022-08-25    16638.596026
2022-08-26    16653.837315
2022-08-29    16492.559544
2022-08-30    16452.364836
2022-08-31    16315.060059
Name: price, Length: 2000, dtype: float64

In [2]:
import pandas as pd

In [6]:
price_df = pd.read_csv("data_ini/price_df.csv",index_col=0,parse_dates=True)
aaa = pd.DataFrame(price_df.shift(-1)['CLOSE'])
aaa['ret_c'] = price_df['CLOSE'].pct_change().shift(-1)
bbb = pd.DataFrame(aaa['ret_c'])

In [10]:
bbb.to_csv("data_ini/test_factor.csv")


aaa
